# Cleaning My Scraped Google Maps Reviews Data

### Imports

In [214]:
import json
import pandas as pd
from pathlib import Path

### Loading JSON files

In [215]:
folder = Path("data/google-maps-review/raw")

destinations = [
    "lisbon",
    "marrakech",
    "hanoi",
    "reykjavik"
]

data = {}

for destination in destinations:

    data[destination] = []

    file_path = folder / f"{destination}.json"

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        data[destination].extend(
            json.load(f)
        )

    # Load second file if it exists
    if destination in [
        "lisbon",
        "marrakech",
        "hanoi"
    ]:

        file_path_1 = (
            folder
            / f"{destination}1.json"
        )

        with open(
            file_path_1,
            "r",
            encoding="utf-8"
        ) as f:

            data[destination].extend(
                json.load(f)
            )

    print(
        destination,
        len(data[destination])
    )

# data = {}

# for destination in destinations:
#     data[destination] = []

#     file_path = folder / f"{destination}.json"

#     with open(file_path, "r", encoding="utf-8") as f:
#         data[destination].extend(json.load(f))

#     print(len(data[destination]))

lisbon 1683
marrakech 2033
hanoi 1912
reykjavik 1912


### Removing entries with errors, converting to df, removing duplicates (if any)

In [216]:
df = {}

for destination in destinations:

    print(
        f"{destination}: original records = "
        f"{len(data[destination])}"
    )

    # Remove entries with errors
    data[destination] = [
        entry
        for entry in data[destination]
        if not entry.get("error")
    ]

    print(
        f"{destination}: after removing errors = "
        f"{len(data[destination])}"
    )

    # Remove unwanted location - agafay desert best camp
    data[destination] = [
        entry
        for entry in data[destination]
        if entry.get("place_name")
        != "Agafay desert best camp"
    ]

    # Convert to DataFrame
    df[destination] = pd.json_normalize(
        data[destination]
    )

    # Remove duplicate reviews
    if "review_id" in df[destination].columns:

        before = len(df[destination])

        df[destination] = df[destination].drop_duplicates(
            subset="review_id",
            keep="first"
        )

        print(
            f"{destination}: removed "
            f"{before - len(df[destination])} "
            f"duplicate reviews"
        )

    print(
        f"{destination}: final records = "
        f"{len(df[destination])}"
    )

lisbon: original records = 1683
lisbon: after removing errors = 1678
lisbon: removed 0 duplicate reviews
lisbon: final records = 1678
marrakech: original records = 2033
marrakech: after removing errors = 2033
marrakech: removed 0 duplicate reviews
marrakech: final records = 1977
hanoi: original records = 1912
hanoi: after removing errors = 1911
hanoi: removed 0 duplicate reviews
hanoi: final records = 1911
reykjavik: original records = 1912
reykjavik: after removing errors = 1912
reykjavik: removed 0 duplicate reviews
reykjavik: final records = 1912


### Checking na values

In [217]:
for destination in destinations:

    print(f"\n{destination.upper()}")

    print(
        df[destination]
        .isna()
        .sum()
    )


LISBON
url                         0
place_id                    0
place_name                  0
country                    86
address                     0
review_id                   0
reviewer_name               0
reviews_by_reviewer         0
photos_by_reviewer        361
reviewer_url                0
local_guide                 0
review_rating               0
review                    819
review_date                 0
number_of_likes             0
response_of_owner        1674
response_date            1674
photos                   1219
review_details            773
profile_pic_url             0
place_general_rating        0
overall_place_riviews       0
questions_answers        1678
fid_location                0
category                  172
cid                         0
timestamp                   0
input.url                   0
input.days_limit            0
input.sort_by               0
dtype: int64

MARRAKECH
url                         0
place_id                    0
place_na

### Cleaning columns and normalising data

In [218]:
columns = [
    "review_id",
    "url",
    "place_id",
    "place_name",
    "country",
    "address",
    "reviewer_name",
    "reviews_by_reviewer",
    "photos_by_reviewer",
    "reviewer_url",
    "local_guide",
    "review_rating",
    "review",
    "review_date",
    "number_of_likes",
    "response_of_owner",
    "response_date",
    "review_details",
    "place_general_rating",
    "overall_place_riviews",
    "questions_answers",
    "category",
    "timestamp"
]

date_columns = [
    "review_date",
    "timestamp"
]

numeric_columns = [
    "reviews_by_reviewer",
    "photos_by_reviewer",
    "review_rating",
    "number_of_likes",
    "place_general_rating",
    "overall_place_riviews"
]

print(df["lisbon"].columns.tolist())

for destination in destinations:
    print(destination, df[destination].shape)

    df[destination] = df[destination][columns].copy()

    df[destination]["destination"] = destination

    for column in date_columns:
        df[destination][column] = pd.to_datetime(
            df[destination][column], 
            errors="coerce"
        )

    for column in numeric_columns:
        df[destination][column] = pd.to_numeric(
            df[destination][column], 
            errors="coerce"
        )

    print(destination, df[destination].shape)

print(df["lisbon"].columns.tolist())
print(df["lisbon"].iloc[0])


['url', 'place_id', 'place_name', 'country', 'address', 'review_id', 'reviewer_name', 'reviews_by_reviewer', 'photos_by_reviewer', 'reviewer_url', 'local_guide', 'review_rating', 'review', 'review_date', 'number_of_likes', 'response_of_owner', 'response_date', 'photos', 'review_details', 'profile_pic_url', 'place_general_rating', 'overall_place_riviews', 'questions_answers', 'fid_location', 'category', 'cid', 'timestamp', 'input.url', 'input.days_limit', 'input.sort_by']
lisbon (1678, 30)
lisbon (1678, 24)
marrakech (1977, 30)
marrakech (1977, 24)
hanoi (1911, 30)
hanoi (1911, 24)
reykjavik (1912, 30)
reykjavik (1912, 24)
['review_id', 'url', 'place_id', 'place_name', 'country', 'address', 'reviewer_name', 'reviews_by_reviewer', 'photos_by_reviewer', 'reviewer_url', 'local_guide', 'review_rating', 'review', 'review_date', 'number_of_likes', 'response_of_owner', 'response_date', 'review_details', 'place_general_rating', 'overall_place_riviews', 'questions_answers', 'category', 'timestam

### Checking all the locations chosen have been scraped

In [219]:
place_names = df["lisbon"]["place_name"].unique()
print(df["lisbon"]["place_name"].unique())
print(len(place_names))
print(
    df["lisbon"]["place_name"]
    .value_counts()
)
# checking if every google maps place name exists in tiktok_locs_count
with open(
    "data/tiktok/processed/tiktoks_locs_count_clean.json",
    "r",
    encoding="utf-8"
) as f:

    locations_count_dict = json.load(f)


all_present = all(
    place_name in locations_count_dict["lisbon"]
    for place_name in place_names
)

print(all_present)


['Tuk Tuk Lisbon Tours' 'Bar Alimentar' 'Machimbombo' 'Ursa Beach'
 'MAAT - Museum of Art, Architecture and Technology'
 'Miradouro da Senhora do Monte' 'Miradouro de São Pedro de Alcântara'
 'Belém Tower' 'Carmo Archaeological Museum' 'Santa Justa Lift'
 'Pink Street' 'Castelo de São Jorge' 'Jerónimos Monastery'
 'Time Out Market' 'Praça do Comércio' 'LX Factory' 'Oceanário de Lisboa'
 'Pastéis de Belém' 'Bessa Restaurante(seafood restaurant ))'
 'National Palace of Pena' 'Miradouro de Santa Catarina']
21
place_name
Pink Street                                          86
Bar Alimentar                                        86
Pastéis de Belém                                     86
Oceanário de Lisboa                                  86
LX Factory                                           86
Praça do Comércio                                    86
Time Out Market                                      86
Jerónimos Monastery                                  86
Castelo de São Jorge         

In [220]:
place_names = df["marrakech"]["place_name"].unique()
print(df["marrakech"]["place_name"].unique())
print(len(place_names))
print(
    df["marrakech"]["place_name"]
    .value_counts()
)
#checking if every google maps place name exists in tiktok_locs_count
with open(
    "data/tiktok/processed/tiktoks_locs_count_clean.json",
    "r",
    encoding="utf-8"
) as f:

    locations_count_dict = json.load(f)


all_present = all(
    place_name in locations_count_dict["marrakech"]
    for place_name in place_names
)

print(all_present)


['Palmeraie Marrakech chameau' 'Marrakech Museum' 'Menara Gardens'
 'Bacha Coffee' 'Souk Semmarine' 'Nomad Marrakech' 'Koutoubia'
 'El Badi Palace' 'La Mamounia' 'Dar El Bacha Museum'
 'Yves Saint Laurent Museum' 'Comptoir Darna' 'Jemaa el-Fnaa'
 'Bahia Palace' 'Jardin Majorelle' 'Le Jardin Secret'
 'Madrasa Ben Youssef' 'Agafay Desert']
18
place_name
Dar El Bacha Museum            119
Madrasa Ben Youssef            119
Menara Gardens                 119
Bacha Coffee                   119
Souk Semmarine                 119
Nomad Marrakech                119
Koutoubia                      119
El Badi Palace                 119
La Mamounia                    119
Marrakech Museum               119
Yves Saint Laurent Museum      119
Comptoir Darna                 119
Jemaa el-Fnaa                  119
Bahia Palace                   119
Jardin Majorelle               119
Le Jardin Secret               119
Agafay Desert                   56
Palmeraie Marrakech chameau     17
Name: count, dty

In [221]:
place_names = df["hanoi"]["place_name"].unique()
print(df["hanoi"]["place_name"].unique())
print(len(place_names))
print(
    df["hanoi"]["place_name"]
    .value_counts()
)
#checking if every google maps place name exists in tiktok_locs_count
with open(
    "data/tiktok/processed/tiktoks_locs_count_clean.json",
    "r",
    encoding="utf-8"
) as f:

    locations_count_dict = json.load(f)


all_present = all(
    place_name in locations_count_dict["hanoi"]
    for place_name in place_names
)

print(all_present)

place_names = set(place_names)

tiktok_places = set(
    locations_count_dict["hanoi"].keys()
)

missing_places = place_names - tiktok_places

print(missing_places)


['St Joseph Cathedral' 'Sky Lotte Observation Deck' 'West Lake'
 'Ho Chi Minh Museum' 'Hoàn Kiếm Lake' 'The Note Coffee' 'Banh Mi 25'
 'Cafe Giảng' 'Train Street' 'Imperial Citadel of Thang Long'
 'Hanoi Old Quarter' 'Bún chả Hương Liên' 'Chợ Hàng Mã'
 'Temple Of Literature' "Ho Chi Minh's Mausoleum" 'Dong xuan market']
16
place_name
Sky Lotte Observation Deck        134
West Lake                         134
Ho Chi Minh Museum                134
Hoàn Kiếm Lake                    134
The Note Coffee                   134
Banh Mi 25                        134
Cafe Giảng                        134
Train Street                      134
Imperial Citadel of Thang Long    134
Hanoi Old Quarter                 134
Bún chả Hương Liên                134
Temple Of Literature              134
Ho Chi Minh's Mausoleum           134
Dong xuan market                  134
Chợ Hàng Mã                        34
St Joseph Cathedral                 1
Name: count, dtype: int64
False
{'Dong xuan market'}


In [222]:
place_names = df["reykjavik"]["place_name"].unique()
# print(df["reykjavik"]["place_name"].unique())
print(len(place_names))
print(
    df["reykjavik"]["place_name"]
    .value_counts()
)
#checking if every google maps place name exists in tiktok_locs_count
with open(
    "data/tiktok/processed/tiktoks_locs_count_clean.json",
    "r",
    encoding="utf-8"
) as f:

    locations_count_dict = json.load(f)


all_present = all(
    place_name in locations_count_dict["reykjavik"]
    for place_name in place_names
)

print(all_present)

place_names = set(place_names)

tiktok_places = set(
    locations_count_dict["reykjavik"].keys()
)

missing_places = place_names - tiktok_places

print(missing_places)


21
place_name
Reynisfjara Beach                           97
Skógafoss                                   97
Blue Lagoon                                 97
Sky Lagoon                                  97
Sun Voyager                                 97
Jökulsárlón                                 97
Thingvellir National Park                   97
FlyOver Iceland                             97
Perlan                                      97
Diamond Beach                               97
Hallgrimskirkja                             97
Black Sand Beach                            97
Skólavörðustígur Rainbow Street             97
Gullfoss Falls                              97
Geysir                                      97
Seljalandsfoss                              97
The Icelandic Phallological Museum          97
Harpa Concert Hall and Conference Centre    97
Lebowski Bar                                97
Grótta Lighthouse                           53
Harbor in Reykjavik                         16

### Turning dfs to csv files (use for anaylsis)

In [223]:
output_folder = Path("data/google-maps-review/processed")

output_folder.mkdir(parents=True, exist_ok=True)

for destination, destination_df in df.items():
    destination_df.to_csv(
        output_folder / f"{destination}_clean.csv",
        index=False
    )

### Turning dfs to excel files (use to view)

In [224]:
output_folder = Path("data/google-maps-review/processed")

output_folder.mkdir(parents=True, exist_ok=True)

output_path = output_folder / "googlemaps_clean.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    for destination, destination_df in df.items():

        # Remove timezone from datetime columns
        # because Excel does not support timezone-aware datetimes
        for column in date_columns:
            destination_df[column] = (
                destination_df[column]
                .dt.tz_localize(None)
            )

        # Write each destination to its own sheet
        destination_df.to_excel(
            writer,
            sheet_name=destination,
            index=False
        )